### PCA 개요 

In [ ]:
# 사이킷런 내장 데이터셋(iris: 붓꽃 데이터) 로드 함수 임포트
from sklearn.datasets import load_iris
import pandas as pd
import matplotlib.pyplot as plt
# 주피터 노트북에서 그래프를 셀 출력 영역에 인라인으로 표시
%matplotlib inline

# 사이킷런 내장 데이터 셋 API 호출
# iris.data: 4개 피처(꽃받침/꽃잎 길이·너비)로 구성된 150개 샘플의 numpy 배열
# iris.target: 0(setosa), 1(versicolor), 2(virginica) 세 가지 품종 레이블
iris = load_iris()

# 넘파이 데이터 셋을 Pandas DataFrame으로 변환
# 4개 피처에 컬럼명 부여 (꽃받침/꽃잎의 길이/너비)
columns = ['sepal_length','sepal_width','petal_length','petal_width']
irisDF = pd.DataFrame(iris.data , columns=columns)
# 타깃(품종) 컬럼 추가
irisDF['target']=iris.target
# 데이터 상위 3개 행 확인
irisDF.head(3)

In [ ]:
# 품종별로 마커 모양을 다르게 지정하여 시각적으로 구분
# setosa는 세모(^), versicolor는 네모(s), virginica는 동그라미(o)
markers=['^', 's', 'o']

# setosa의 target 값은 0, versicolor는 1, virginica는 2. 각 target 별로 다른 shape으로 scatter plot
# enumerate를 사용해 인덱스 i(0,1,2)와 marker를 함께 순회
for i, marker in enumerate(markers):
    # 해당 품종(target == i)에 속하는 데이터만 필터링하여 x, y축 데이터 추출
    x_axis_data = irisDF[irisDF['target']==i]['sepal_length']
    y_axis_data = irisDF[irisDF['target']==i]['sepal_width']
    # 산점도 그리기 (label에 품종명을 부여하여 범례에 표시)
    plt.scatter(x_axis_data, y_axis_data, marker=marker,label=iris.target_names[i])

# 품종별 범례 표시
plt.legend()
plt.xlabel('sepal length')
plt.ylabel('sepal width')
# 그래프 표시: PCA 적용 전 원본 데이터에서 sepal 두 개의 피처만 사용했을 때의 분포 확인
plt.show()

In [ ]:
# StandardScaler: 각 피처를 평균 0, 표준편차 1인 표준정규분포 형태로 변환
# PCA는 분산이 큰 방향을 주성분으로 찾기 때문에, 피처 간 스케일이 다르면
# 스케일이 큰 피처에 결과가 편향됨 → PCA 전 표준화는 필수 전처리
from sklearn.preprocessing import StandardScaler

# Target 값을 제외한 모든 속성 값을 StandardScaler를 이용하여 표준 정규 분포를 가지는 값들로 변환
# iloc[:, :-1]: 모든 행, 마지막 컬럼(target) 제외 → 4개 피처만 선택
# fit_transform: 학습(평균/표준편차 계산)과 변환을 한 번에 수행
iris_scaled = StandardScaler().fit_transform(irisDF.iloc[:, :-1])

In [ ]:
# PCA(Principal Component Analysis): 주성분 분석
# - 고차원 데이터를 분산이 가장 큰 축(주성분)을 찾아 저차원으로 축소
# - 차원 축소를 통해 시각화, 노이즈 제거, 학습 성능 향상 등을 도모
from sklearn.decomposition import PCA

# 2개의 주성분으로 차원 축소 수행 (4차원 → 2차원)
pca = PCA(n_components=2)

# fit(): 입력 데이터로부터 주성분(고유벡터)을 학습
# transform(): 학습된 주성분 축으로 데이터를 투영(projection)
pca.fit(iris_scaled)
iris_pca = pca.transform(iris_scaled)
# 변환 후 shape 확인: (150, 2) - 샘플 수는 동일하지만 피처가 4개에서 2개로 축소됨
print(iris_pca.shape)

In [ ]:
# PCA 변환된 데이터의 컬럼명을 각각 pca_component_1, pca_component_2로 명명
# - pca_component_1: 데이터 분산을 가장 잘 설명하는 첫 번째 주성분
# - pca_component_2: 첫 번째 주성분에 직교하면서 두 번째로 분산을 잘 설명하는 주성분
pca_columns=['pca_component_1','pca_component_2']
# PCA 결과(numpy 배열)를 DataFrame으로 변환
irisDF_pca = pd.DataFrame(iris_pca, columns=pca_columns)
# 원본 target을 같은 행 순서로 다시 결합 (시각화 및 분류 비교용)
irisDF_pca['target']=iris.target
irisDF_pca.head(3)

In [ ]:
# PCA 변환 후 2차원 공간에서 품종별 분포를 시각화
# setosa는 세모, versicolor는 네모, virginica는 동그라미로 표시
markers=['^', 's', 'o']

# pca_component_1을 x축, pca_component_2를 y축으로 scatter plot 수행
# - 원본 4차원 데이터의 정보를 최대한 보존하면서 2차원으로 압축한 결과
# - sepal_length/width만 사용했을 때보다 품종 간 구분이 더 잘 되는지 확인
for i, marker in enumerate(markers):
    x_axis_data = irisDF_pca[irisDF_pca['target']==i]['pca_component_1']
    y_axis_data = irisDF_pca[irisDF_pca['target']==i]['pca_component_2']
    plt.scatter(x_axis_data, y_axis_data, marker=marker,label=iris.target_names[i])

plt.legend()
plt.xlabel('pca_component_1')
plt.ylabel('pca_component_2')
plt.show()

In [ ]:
# explained_variance_ratio_: 각 주성분이 전체 분산을 설명하는 비율
# - 예: [0.72, 0.23]이면 첫 번째 주성분이 72%, 두 번째가 23%를 설명 → 두 성분이 합쳐 약 95% 정보 보존
# - 이 값으로 차원 축소 후에도 원본 정보가 얼마나 유지되는지 정량적으로 판단 가능
print(pca.explained_variance_ratio_)

In [ ]:
# PCA 변환 전후의 분류 성능 비교를 위한 라이브러리 임포트
from sklearn.ensemble import RandomForestClassifier
# cross_val_score: 교차 검증으로 모델의 평균 성능을 측정해 일반화 능력을 평가
from sklearn.model_selection import cross_val_score
import numpy as np

# 랜덤포레스트 분류기 생성 (random_state=156으로 결과 재현성 보장)
rcf = RandomForestClassifier(random_state=156)
# 원본 4차원 iris 데이터에 대해 3-fold 교차 검증으로 정확도 측정
scores = cross_val_score(rcf, iris.data, iris.target,scoring='accuracy',cv=3)
# 각 fold별 정확도와 평균 정확도 출력
print('원본 데이터 교차 검증 개별 정확도:',scores)
print('원본 데이터 평균 정확도:', np.mean(scores))

In [ ]:
# PCA로 2차원으로 축소된 데이터만 추출하여 학습 입력으로 사용
pca_X = irisDF_pca[['pca_component_1', 'pca_component_2']]
# 동일한 랜덤포레스트 모델로 PCA 변환 데이터에 대해 3-fold 교차 검증 수행
scores_pca = cross_val_score(rcf, pca_X, iris.target, scoring='accuracy', cv=3 )
print('PCA 변환 데이터 교차 검증 개별 정확도:',scores_pca)
print('PCA 변환 데이터 평균 정확도:', np.mean(scores_pca))
# → 4차원 → 2차원으로 절반의 정보로 축소했음에도 정확도가 크게 떨어지지 않음을 확인
# → PCA가 차원 축소 후에도 분류에 필요한 정보를 잘 보존함을 보여주는 사례

* credit card 데이터 세트 PCA 변환

In [ ]:
# 신용카드 디폴트(default) 예측 데이터셋 로드
# header=1: 첫 행은 카테고리 헤더라 의미 없으므로 두 번째 행(인덱스 1)을 컬럼명으로 사용
# iloc[0:, 1:]: 모든 행, 첫 번째 컬럼(ID)을 제외하고 가져옴 (ID는 분석에 불필요)
import pandas as pd

df = pd.read_excel('pca_credit_card.xls', header=1, sheet_name='Data').iloc[0:,1:]
# 데이터 shape 확인 (행 수, 컬럼 수)
print(df.shape)
# 데이터 구조 확인
df.head(3)

In [ ]:
# 컬럼명 정리:
# - 'PAY_0' → 'PAY_1': 다른 PAY_2~PAY_6과 명명 일관성을 맞추기 위함
# - 'default payment next month' → 'default': 긴 타깃명을 짧게 단순화
# inplace=True: 원본 DataFrame을 직접 수정
df.rename(columns={'PAY_0':'PAY_1','default payment next month':'default'}, inplace=True)
# 타깃(y)과 피처(X) 분리: default(1=연체, 0=정상)를 예측
y_target = df['default']
# axis=1: 컬럼 방향으로 default 컬럼 제거 → 나머지 컬럼이 모두 피처
X_features = df.drop('default', axis=1)

In [ ]:
# 피처 데이터의 정보 확인
# - 각 컬럼의 데이터 타입(dtype), 결측치(non-null count), 메모리 사용량 등을 출력
# - 데이터 전처리 전에 데이터 품질을 확인하는 기본 단계
X_features.info()

In [ ]:
# 시각화 라이브러리 임포트
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

# 피처 간 상관계수(피어슨 상관계수) 행렬 계산
# - 상관계수가 1에 가까우면 강한 양의 선형 관계, -1에 가까우면 강한 음의 관계
# - PCA 적용 후보 컬럼군(BILL_AMT, PAY 등)을 찾기 위해 상관관계가 높은 그룹을 확인
corr = X_features.corr()
# 히트맵 크기 설정 (피처 수가 많아 큰 사이즈로)
plt.figure(figsize=(14,14))
# annot=True: 각 셀에 상관계수 값을 표시
# fmt='.1g': 유효 숫자 1자리로 간결하게 표시
sns.heatmap(corr, annot=True, fmt='.1g')


In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# 상관관계가 높은 BILL_AMT1 ~ BILL_AMT6 (6개월 청구액) 컬럼명을 list comprehension으로 생성
cols_bill = ['BILL_AMT'+str(i) for i in range(1,7)]
# PAY_1 ~ PAY_6 (6개월 결제 상태) 컬럼명 생성
cols_pay = ['PAY_' + str(i) for i in range(1, 7)]
# PAY_AMT1 ~ PAY_AMT6 (6개월 결제 금액) 컬럼명 생성
cols_amt = ['PAY_AMT' + str(i) for i in range(1, 7)]
print(cols_bill)
# cols_bill 리스트에 PAY, PAY_AMT 컬럼명까지 모두 추가 (총 18개의 컬럼)
cols_bill.extend(cols_pay)
cols_bill.extend(cols_amt)
print('대상 속성명:',cols_bill)

# 2개의 PCA 속성을 가진 PCA 객체 생성하고, explained_variance_ratio_ 계산 위해 fit() 호출
# PCA 적용 전 표준화: 각 컬럼의 스케일이 다르므로 표준화로 동일 스케일로 맞춤
scaler = StandardScaler()
df_cols_scaled = scaler.fit_transform(X_features[cols_bill])
# 표준화된 값을 다시 원본 DataFrame에 덮어쓰기 (이후 cell-16에서 사용)
X_features.loc[:, cols_bill] = df_cols_scaled
# 18개 상관 피처를 2개 주성분으로 압축
pca = PCA(n_components=2)
pca.fit(df_cols_scaled)
# 두 주성분이 18개 피처의 분산을 얼마나 설명하는지 출력
# - BILL_AMT 그룹은 상관관계가 매우 높아 첫 주성분 하나로 대부분의 분산이 설명됨
print('PCA Component별 변동성:', pca.explained_variance_ratio_)

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# 원본 전체 피처(23개)에 대해 랜덤포레스트로 분류 성능 측정 (베이스라인)
# n_estimators=300: 트리 300개로 앙상블
rcf = RandomForestClassifier(n_estimators=300, random_state=156)
# 3-fold 교차 검증 수행
scores = cross_val_score(rcf, X_features, y_target, scoring='accuracy', cv=3 )

# 각 fold별 정확도와 평균 정확도 출력
print('CV=3 인 경우의 개별 Fold세트별 정확도:',scores)
print('평균 정확도:{0:.4f}'.format(np.mean(scores)))


In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# 원본 데이터셋(23개 피처 전체)에 먼저 StandardScaler 적용
# PCA는 분산 기반이므로 스케일 정규화가 결과에 큰 영향을 미침
scaler = StandardScaler()
df_scaled = scaler.fit_transform(X_features)

# 6개의 Component를 가진 PCA 변환을 수행하고 cross_val_score()로 분류 예측 수행
# 23개 피처 → 6개 주성분으로 압축 (약 74% 차원 축소)
pca = PCA(n_components=6)
df_pca = pca.fit_transform(df_scaled)
# 동일 모델(rcf)로 PCA 변환 데이터에 대해 3-fold 교차 검증
scores_pca = cross_val_score(rcf, df_pca, y_target, scoring='accuracy', cv=3)

print('CV=3 인 경우의 PCA 변환된 개별 Fold세트별 정확도:',scores_pca)
print('PCA 변환 데이터 셋 평균 정확도:{0:.4f}'.format(np.mean(scores_pca)))
# → 23개 피처를 6개로 축소했음에도 원본과 유사한 정확도를 유지
# → 차원 축소로 학습 시간/메모리는 절감하면서 성능은 거의 손실되지 않음을 확인